In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

os.makedirs("knn", exist_ok=True)

In [2]:
CLASSIFIED_DIR = "clean_crop_contribution_data/aggregated_data/kmeans_classes"
FEATURES_FILE  = "engineered_climate_features_slight_correlation.csv"
MIN_ROWS       = 200

csv_files = glob.glob(os.path.join(CLASSIFIED_DIR, "*_classified.csv"))
print(f"Found {len(csv_files)} classified files")

Found 54 classified files


In [3]:
features_df = pd.read_csv(FEATURES_FILE)

In [4]:
all_crop_summary = []
N_SPLITS = 5

for csv_path in csv_files:
    crop_name = os.path.basename(csv_path).replace("_classified.csv", "")
    print(f"\n{'='*60}")
    print(f"CROP: {crop_name}")
    print(f"{'='*60}")

    # ── Load & gate on row count ──────────────────────────────────
    class_df = pd.read_csv(csv_path)
    if len(class_df) <= MIN_ROWS:
        print(f"  Skipping — only {len(class_df)} rows (need > {MIN_ROWS})")
        continue

    # ── Merge with climate features ───────────────────────────────
    class_df["location"] = class_df["State"] + "_" + class_df["District"]
    merged_df = class_df.merge(features_df, on="location", how="inner")
    print(f"  Merged shape: {merged_df.shape}")

    # ── Drop classes with too few samples to survive CV ──────────
    class_counts_raw = merged_df["Yield_Class"].value_counts()
    valid_classes    = class_counts_raw[class_counts_raw >= N_SPLITS].index.tolist()
    dropped_classes  = class_counts_raw[class_counts_raw <  N_SPLITS].index.tolist()

    if dropped_classes:
        print(f"\n  !!! Dropping classes with < {N_SPLITS} samples: {dropped_classes}")
        merged_df = merged_df[merged_df["Yield_Class"].isin(valid_classes)].copy()
        print(f"  Rows after dropping: {len(merged_df)}")

    if len(merged_df) <= MIN_ROWS:
        print(f"  Skipping — too few rows after class drop ({len(merged_df)})")
        continue

    if len(valid_classes) < 2:
        print(f"  Skipping — fewer than 2 valid classes remain")
        continue

    # ── Class counts & proportions ────────────────────────────────
    class_counts = merged_df["Yield_Class"].value_counts()
    class_props  = merged_df["Yield_Class"].value_counts(normalize=True)
    print("\n  Class counts:\n", class_counts.to_string())
    print("\n  Class proportions:\n", class_props.round(3).to_string())

    # ── Encode target ─────────────────────────────────────────────
    le = LabelEncoder()
    merged_df["Yield_Class_Encoded"] = le.fit_transform(merged_df["Yield_Class"])
    encoding_map = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"\n  Encoding: {encoding_map}")

    # ── Build X, y ────────────────────────────────────────────────
    drop_cols = ["State", "District", "location", "Median", "Max", "Yield_Class"]
    X = merged_df.drop(columns=drop_cols + ["Yield_Class_Encoded"])
    y = merged_df["Yield_Class_Encoded"]

    y = pd.Series(LabelEncoder().fit_transform(y), index=y.index)
    n_classes = len(np.unique(y))
    print(f"  Unique classes in y after re-encode: {np.unique(y).tolist()}")

    # ── Sample weights (computed but not passed to KNN fit) ───────
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    class_weight_dict = dict(zip(classes, weights))
    sample_weights = y.map(class_weight_dict)

    # ── Cross-validation ──────────────────────────────────────────
    kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    fold_metrics         = []
    shap_importance_list = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        print(f"\n  ---- Fold {fold+1} ----")

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # KNN requires feature scaling
        scaler         = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled   = scaler.transform(X_val)

        model = KNeighborsClassifier(
            n_neighbors=5,
            weights="distance",
            metric="minkowski"
        )
        # Note: KNN does not support sample_weight in fit()
        model.fit(X_train_scaled, y_train)

        preds = model.predict(X_val_scaled)
        acc   = accuracy_score(y_val, preds)
        f1    = f1_score(y_val, preds, average="weighted")
        fold_metrics.append((acc, f1))
        print(f"    Accuracy: {acc:.4f}  |  F1: {f1:.4f}")

        # SHAP — KernelExplainer (no TreeExplainer for KNN)
        background  = shap.sample(X_train_scaled, 50)
        explainer   = shap.KernelExplainer(model.predict_proba, background)
        shap_values = explainer.shap_values(X_val_scaled, nsamples=100)

        if isinstance(shap_values, list):
            shap_vals = np.mean([np.abs(sv) for sv in shap_values], axis=0)
        elif shap_values.ndim == 3:
            shap_vals = np.abs(shap_values).mean(axis=2)
        else:
            shap_vals = np.abs(shap_values)

        shap_importance_list.append(shap_vals.mean(axis=0))

    # ── Aggregate CV metrics ──────────────────────────────────────
    fold_metrics = np.array(fold_metrics)
    acc_mean, acc_std = fold_metrics[:, 0].mean(), fold_metrics[:, 0].std()
    f1_mean,  f1_std  = fold_metrics[:, 1].mean(), fold_metrics[:, 1].std()

    print(f"\n  CV Accuracy : {acc_mean:.4f}, SD: {acc_std:.4f}")
    print(f"  CV F1       : {f1_mean:.4f}, SD: {f1_std:.4f}")

    # ── SHAP importance ───────────────────────────────────────────
    shap_importance = np.mean(shap_importance_list, axis=0)
    if shap_importance.ndim > 1:
        shap_importance = np.abs(shap_importance).mean(
            axis=tuple(range(shap_importance.ndim - 1))
        )

    importance_df = pd.DataFrame({
        "feature":    X.columns,
        "importance": shap_importance
    }).sort_values("importance", ascending=False)

    # ── Save txt report ───────────────────────────────────────────
    txt_path = os.path.join("knn", f"{crop_name}_results.txt")
    with open(txt_path, "w") as f:
        f.write(f"CROP: {crop_name}\n")
        f.write(f"Total rows after merge: {merged_df.shape[0]}\n\n")

        if dropped_classes:
            f.write(f"DROPPED CLASSES (< {N_SPLITS} samples): {dropped_classes}\n\n")

        f.write("CLASS COUNTS\n")
        f.write(class_counts.to_string() + "\n\n")
        f.write("CLASS PROPORTIONS\n")
        f.write(class_props.round(4).to_string() + "\n\n")

        f.write("ENCODING\n")
        f.write(str(encoding_map) + "\n\n")

        f.write("FOLD-WISE METRICS\n")
        for i, (a, fi) in enumerate(fold_metrics, 1):
            f.write(f"  Fold {i}: Accuracy={a:.4f}  F1={fi:.4f}\n")
        f.write("\n")

        f.write("AGGREGATED CV METRICS\n")
        f.write(f"  Accuracy : {acc_mean:.4f} ± {acc_std:.4f}\n")
        f.write(f"  F1       : {f1_mean:.4f} ± {f1_std:.4f}\n\n")

        f.write("SHAP FEATURE IMPORTANCE (sorted)\n")
        f.write(importance_df.to_string(index=False) + "\n")

    print(f"  Report saved → {txt_path}")

    # ── SHAP bar plot ─────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 10))
    importance_df.head(20).plot(
        x="feature", y="importance", kind="barh", ax=ax, legend=False
    )
    ax.invert_yaxis()
    ax.set_title(f"{crop_name} — Top 20 SHAP Features")
    ax.set_xlabel("Mean |SHAP value|")
    plt.tight_layout()
    plot_path = os.path.join("knn", f"{crop_name}_shap.png")
    plt.savefig(plot_path, dpi=150)
    plt.close()
    print(f"  SHAP plot saved → {plot_path}")

    # ── Collect for cross-crop summary ────────────────────────────
    all_crop_summary.append({
        "crop":      crop_name,
        "n_rows":    merged_df.shape[0],
        "n_classes": n_classes,
        "dropped":   ", ".join(dropped_classes) if dropped_classes else "none",
        "acc_mean":  acc_mean,
        "acc_std":   acc_std,
        "f1_mean":   f1_mean,
        "f1_std":    f1_std,
    })


CROP: arecanut
  Skipping — only 152 rows (need > 200)

CROP: arhar_tur
  Merged shape: (665, 42)

  Class counts:
 Yield_Class
Medium    293
High      275
Low        97

  Class proportions:
 Yield_Class
Medium    0.441
High      0.414
Low       0.146

  Encoding: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}
  Unique classes in y after re-encode: [0, 1, 2]

  ---- Fold 1 ----
    Accuracy: 0.6692  |  F1: 0.6680


  0%|          | 0/133 [00:00<?, ?it/s]


  ---- Fold 2 ----
    Accuracy: 0.6617  |  F1: 0.6567


  0%|          | 0/133 [00:00<?, ?it/s]


  ---- Fold 3 ----
    Accuracy: 0.7068  |  F1: 0.7018


...


  ---- Fold 4 ----
    Accuracy: 0.7540  |  F1: 0.7583


  0%|          | 0/126 [00:00<?, ?it/s]


  ---- Fold 5 ----
    Accuracy: 0.7302  |  F1: 0.7273


  0%|          | 0/126 [00:00<?, ?it/s]


  CV Accuracy : 0.7238, SD: 0.0169
  CV F1       : 0.7255, SD: 0.0173
  Report saved → knn\wheat_results.txt
  SHAP plot saved → knn\wheat_shap.png


In [5]:
summary_df = pd.DataFrame(all_crop_summary).sort_values("f1_mean", ascending=False)

print("\n" + "="*60)
print("CROSS-CROP SUMMARY")
print("="*60)
print(summary_df.to_string(index=False))

print("\nOverall average across all crops:")
print(f"  Accuracy : {summary_df['acc_mean'].mean():.4f} ± {summary_df['acc_std'].mean():.4f}")
print(f"  F1       : {summary_df['f1_mean'].mean():.4f} ± {summary_df['f1_std'].mean():.4f}")

summary_df.to_csv("knn/all_crops_summary.csv", index=False)
print("\nSummary saved → knn/all_crops_summary.csv")


CROSS-CROP SUMMARY
                crop  n_rows  n_classes dropped  acc_mean  acc_std  f1_mean   f1_std
           sugarcane     663          3    none  0.865789 0.012660 0.864962 0.016095
      other_oilseeds     223          3    none  0.865455 0.020158 0.862225 0.018807
               mesta     258          3    none  0.852941 0.044242 0.846143 0.055844
        sweet_potato     457          3    none  0.844529 0.035413 0.842582 0.034734
               onion     573          3    none  0.839481 0.027092 0.839628 0.025994
      peas_and_beans     518          3    none  0.830097 0.008108 0.826155 0.012940
        cowpea_lobia     231          3    none  0.814246 0.055702 0.809688 0.068976
           groundnut     583          2    High  0.809608 0.044396 0.808611 0.045050
              garlic     439          3    none  0.804075 0.034901 0.804472 0.036040
              potato     600          3    none  0.805000 0.023921 0.801406 0.024261
            turmeric     502          3    no